In [2]:
!pip install sentence-transformers faiss-cpu google-generativeai

In [3]:
import pandas as pd
import numpy as np
import faiss
import time

from sentence_transformers import SentenceTransformer

import google.generativeai as genai

from google.colab import userdata

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [5]:
GOOGLE_API_KEY = userdata.get(
    "Research++"
)

genai.configure(
    api_key=GOOGLE_API_KEY
)

llm = genai.GenerativeModel(
    "gemini-2.5-flash"
)

print(
    "Gemini configured successfully."
)

Gemini configured successfully.


In [14]:
from google.colab import files

uploaded = files.upload()
uploaded = files.upload()

Saving papers.json to papers (1).json


Saving researchforge_faiss.index to researchforge_faiss.index


In [15]:
df = pd.read_json(
    "papers.json"
)

faiss_index = faiss.read_index(
    "researchforge_faiss.index"
)

print(
    f"Loaded {len(df)} papers."
)

Loaded 100 papers.


In [16]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print(
    "Embedding model ready."
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model ready.


In [17]:
def retrieve_papers(
    user_query,
    top_k=10
):

    query_embedding = embedding_model.encode(
        [user_query]
    )

    distances, indices = faiss_index.search(
        np.array(
            query_embedding
        ).astype("float32"),
        k=top_k
    )

    retrieved_papers = []

    for idx in indices[0]:

        retrieved_papers.append({

            "title":
            df.iloc[idx]["title"],

            "abstract":
            df.iloc[idx]["abstract"],

            "published":
            df.iloc[idx]["published"]

        })

    return retrieved_papers

In [18]:
def generate_research_ideas(
    user_query,
    retrieved_papers
):

    evidence_context = ""

    for paper in retrieved_papers:

        evidence_context += f"""

Title:
{paper['title']}

Abstract:
{paper['abstract']}

"""

    prompt = f"""

You are an AI research engineer.

User Question:

{user_query}

Retrieved Scientific Evidence:

{evidence_context}

Task:

Generate practical,
evidence-grounded
research innovation ideas.

Return:

1. Current state of the field

2. Major limitations

3. Underexplored opportunities

4. 2–3 realistic research ideas

For each idea include:

- core intuition
- why it is different
- possible implementation
- evaluation approach
- expected risks

Keep proposals technically realistic.

Avoid overly futuristic
or vague ideas.

Use retrieved evidence only.

"""

    for attempt in range(3):

        try:

            response = llm.generate_content(
                prompt
            )

            return response.text

        except Exception as e:

            print(
                f"Retry {attempt+1}/3 after API limit..."
            )

            time.sleep(60)

    return (
        "Generation failed after retries."
    )

In [19]:
query = """
Propose practical research ideas
for multimodal deepfake detection.
"""

papers = retrieve_papers(
    query
)

idea_output = generate_research_ideas(
    query,
    papers
)

print(
    idea_output
)

## 1. Current State of the Field

Multimodal deepfake detection, specifically involving audio and visual streams, is an increasingly crucial area of research due to the rapid advancements in generative AI capable of fabricating convincing speech and video (Abstract 3, 6). While historically deepfake detection focused on single modalities (audio *or* video), there is a strong consensus on the need for multimodal detectors that can cope with modern deepfakes manipulating both (Abstract 1, 2, 5, 10).

Current approaches include:
*   **Fusion-based Methods**: Integrating audio-visual features, often by combining labels specific to each single modality (Abstract 1) or fusing streams at various levels (Abstract 4, 5). Some leverage advanced feature extraction techniques like facial characteristics and mel-spectrograms (Abstract 5).
*   **Large Multimodal Models (LMMs)**: Pilot studies show supervised fine-tuned LMMs (e.g., AV-LMMDetect built on Qwen 2.5 Omni) can achieve state-of-the-art per